# Getting started with cuPIQP

The cuPIQP is a GPU-accelerated **proximal interior-point** solver for convex **quadratic programs (QPs)** of the form

$$
\begin{aligned}
\min_{x}\quad & \tfrac{1}{2}\, x^\top P x + c^\top x \\
\mathrm{s.t}\quad & A x = b, \\
                  & h_l \le G x \le h_u, \\
                  & x_l \le x \le x_u,
\end{aligned}
$$

where

| symbol | meaning | shape |
|---|---|---|
| $P = P^\top \succeq 0$ | quadratic cost (symmetric positive semidefinite) | $n \times n$ |
| $c$ | linear cost | $n$ |
| $A,\, b$ | equality constraints | $p \times n$, $\;p$ |
| $G,\, h_l,\, h_u$ | two-sided inequality constraints | $m \times n$, $\;m$, $\;m$ |
| $x_l,\, x_u$ | element-wise box bounds on $x$ | $n$, $\;n$ |

**Unbounded entries** (one-sided constraints, free variables) are set to $\pm\infty$
(use `cupy.inf`); cuPIQP detects these and drops the corresponding rows/bounds
automatically.

All problem data lives **on the GPU**: dense arrays as `cupy` arrays, or sparse
matrices as `cupyx.scipy.sparse` CSR. This notebook has two parts:

1. **A single QP** — solved with both the dense and the sparse backend.
2. **A batch of QPs** — cuPIQP is *natively batched*: every array's **leading
   dimension is the batch size** $B$, and the whole batch is solved in one GPU call.


In [ ]:
import cupy as cp
from cupyx.scipy.sparse import csr_matrix

from cupiqp import DenseSolver, SparseSolver, Status

# Part 1 — Solving a single QP

## Define the problem data

We solve the small two-variable QP

$$
\begin{aligned}
\min_{x_1,\,x_2}\quad & \tfrac12\bigl(6 x_1^2 + 4 x_2^2\bigr) - x_1 - 4 x_2 \\
\text{s.t.}\quad
  & x_1 - 2 x_2 = 1, \\
  & -10 \le x_1 - x_2 \le 0.2, \\
  & 2 x_1 \le -1, \\
  & x_1 \le 1, \\
  & x_2 \ge -1.
\end{aligned}
$$

Note the **one-sided** pieces: the inequality $2 x_1 \le -1$ has no lower bound
($h_l = -\infty$), and each variable is bounded on only one side. We build everything
**directly as cupy arrays** so the data already lives on the GPU.

In [ ]:
# quadratic + linear cost
P = cp.array([[6.0, 0.0],
              [0.0, 4.0]])
c = cp.array([-1.0, -4.0])

# equality constraint:  A x = b
A = cp.array([[1.0, -2.0]])
b = cp.array([1.0])

# two-sided inequalities:  h_l <= G x <= h_u   (use -inf / +inf for one-sided)
G   = cp.array([[1.0, -1.0],
                [2.0,  0.0]])
h_l = cp.array([-10.0, -cp.inf])
h_u = cp.array([  0.2,  -1.0])

# box bounds:  x_l <= x <= x_u
x_l = cp.array([-cp.inf, -1.0])
x_u = cp.array([   1.0,  cp.inf])

## Dense backend

`DenseSolver` works with **dense** cupy arrays for `P`, `A`, `G`. Create the solver,
optionally tweak `solver.settings`, then `setup(...)` the problem and `solve()`.
The solution and per-problem info are exposed through `solver.result`.

In [ ]:
solver = DenseSolver()
solver.settings.verbose = True        # print the banner + interior-point iteration log

solver.setup(P=P, c=c, A=A, b=b, G=G, h_l=h_l, h_u=h_u, x_l=x_l, x_u=x_u)
solver.solve()

# result.x carries a leading batch dimension (B, n); here B = 1
x_dense = solver.result.x.get()[0]
print("status  :", solver.result.info.status[0].name)
print("solution:", x_dense)

## Sparse backend

`SparseSolver` expects `P`, `A`, `G` as **GPU CSR** matrices
(`cupyx.scipy.sparse.csr_matrix`); the vectors stay as cupy arrays. We reuse the exact
same data, just wrapping the matrices as CSR. For larger, structurally sparse problems
this is far more efficient than the dense backend.

In [ ]:
solver = SparseSolver()
solver.settings.verbose = True

solver.setup(
    P=csr_matrix(P), c=c,
    A=csr_matrix(A), b=b,
    G=csr_matrix(G), h_l=h_l, h_u=h_u,
    x_l=x_l, x_u=x_u,
)
solver.solve()

x_sparse = solver.result.x.get()[0]
print("status  :", solver.result.info.status[0].name)
print("solution:", x_sparse)

In [ ]:
assert solver.result.info.status[0] == Status.CUPIQP_SOLVED
assert cp.allclose(cp.asarray(x_dense), cp.asarray(x_sparse), atol=1e-6)
print(f"dense  solution: {x_dense}")
print(f"sparse solution: {x_sparse}")
print("Both backends converged to the same optimum.")

# Part 2 — Solving a batch of QPs

cuPIQP is **natively batched**: it solves $B$ independent QPs in a *single* GPU call.
The only rule is that **the batch size is the leading dimension** of every array:

| array | single | batched |
|---|---|---|
| `P` | `(n, n)` | `(B, n, n)` |
| `c`, `x_l`, `x_u` | `(n,)` | `(B, n)` |
| `A` / `G` | `(p, n)` / `(m, n)` | `(B, p, n)` / `(B, m, n)` |
| `b`, `h_l`, `h_u` | `(p,)` / `(m,)` | `(B, p)` / `(B, m)` |

`solver.result.x` then has shape `(B, n)` and `solver.result.info.status` is a list of
`B` statuses (one per problem).

Here we reuse the **same QP structure** from Part 1 but give each problem a different
**equality target** `b` — like solving the same controller for several set-points at
once. Problem `1` (`b = 1.0`) reproduces the single-problem solution.

## Build the batch

In [ ]:
B = 4

# only the equality target b differs across the batch; everything else is shared
b_batch = cp.array([[0.9], [1.0], [1.1], [1.2]])      # shape (B, p) with p = 1

# replicate the shared data along the leading batch dimension -> (B, ...)
stack = lambda M: cp.stack([M] * B)
P_b, c_b, A_b, G_b = stack(P), stack(c), stack(A), stack(G)
h_l_b, h_u_b, x_l_b, x_u_b = stack(h_l), stack(h_u), stack(x_l), stack(x_u)

print("batched shapes:", "P", P_b.shape, "c", c_b.shape, "A", A_b.shape, "b", b_batch.shape)

## Dense backend (batched)

Exactly the same call as Part 1 — just hand `setup` the batched `(B, ...)` arrays.

In [ ]:
dense_solver = DenseSolver()
dense_solver.settings.verbose = True

dense_solver.setup(P=P_b, c=c_b, A=A_b, b=b_batch, G=G_b,
                   h_l=h_l_b, h_u=h_u_b, x_l=x_l_b, x_u=x_u_b)
dense_solver.solve()

X_dense = dense_solver.result.x.get()                 # (B, n)
print()
for i, st in enumerate(dense_solver.result.info.status):
    print(f"problem {i}:  b = {float(b_batch[i, 0]):.1f}   status = {st.name}   x = {X_dense[i]}")

## Sparse backend (batched)

For the sparse backend, pass `P`, `A`, `G` as **lists of `B` CSR matrices** that share
the same sparsity pattern; the vectors are stacked `(B, ...)` just like the dense case.
(Here every problem shares the same matrices, so the pattern is trivially identical.)

In [ ]:
sparse_solver = SparseSolver()
sparse_solver.settings.verbose = True

sparse_solver.setup(
    P=[csr_matrix(P)] * B, c=c_b,
    A=[csr_matrix(A)] * B, b=b_batch,
    G=[csr_matrix(G)] * B, h_l=h_l_b, h_u=h_u_b,
    x_l=x_l_b, x_u=x_u_b,
)
sparse_solver.solve()

X_sparse = sparse_solver.result.x.get()
print()
for i, st in enumerate(sparse_solver.result.info.status):
    print(f"problem {i}:  b = {float(b_batch[i, 0]):.1f}   status = {st.name}   x = {X_sparse[i]}")

In [ ]:
assert all(st == Status.CUPIQP_SOLVED for st in dense_solver.result.info.status)
assert cp.allclose(cp.asarray(X_dense), cp.asarray(X_sparse), atol=1e-6)
print("All problems solved; dense and sparse batches agree.")